# RouterArena epoch 1-3 예측 그룹 분포 -- 건수 포함 재시각화

이 notebook은 새로 학습하거나 추론을 다시 실행하지 않습니다. `modernbert_full_colab.ipynb`가
이미 만든 epoch 1~3 RouterArena 평가 결과(`epoch-01/02/03/metrics.json`)를 Google Drive에서
그대로 읽어, 퍼센티지만 있던 기존 predicted-group heatmap/stacked bar를 **건수(count)**까지
함께 표시하도록 다시 그립니다.

- GPU가 필요 없습니다. Colab CPU runtime으로 충분합니다.
- `metrics.json`에는 이미 `routerarena_labels_within_predicted_groups.groups[group].counts`가
  들어 있으므로 재추론 없이 시각화 함수만 다시 호출하면 됩니다.
- 참고: 이 notebook을 만든 로컬 세션에서 마운트돼 있던 Google Drive 계정
  (`alexyoo0506@gmail.com`)에는 `LLM_Router` 학습 결과 폴더가 없어서, 실제 Colab에서 학습에
  쓴 계정/경로를 여기서 직접 확인하지 못했습니다. 아래 2번째 셀이 후보 폴더 목록을 출력하니,
  `modernbert-router-balanced-v2`와 이름이 다르면 3번째 셀의 `DRIVE_RUN`을 실제 이름으로
  고쳐서 실행하세요.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
TRAINING_ROOT = Path("/content/drive/MyDrive/LLM_Router/training")
if not TRAINING_ROOT.is_dir():
    raise FileNotFoundError(
        f"{TRAINING_ROOT} 이 없습니다. Drive 계정과 경로를 확인하세요."
    )
print("Available run directories under", TRAINING_ROOT, ":")
for path in sorted(TRAINING_ROOT.iterdir()):
    print(" -", path.name)

In [ ]:
# modernbert_full_colab.ipynb가 쓰는 것과 같은 run 이름입니다.
# 위 셀의 목록에 이 이름이 없으면 실제 이름으로 바꾸세요.
DRIVE_RUN = TRAINING_ROOT / "modernbert-router-balanced-v2"
DRIVE_EVALUATION_ROOT = DRIVE_RUN / "routerarena-epoch-evaluations"
if not DRIVE_EVALUATION_ROOT.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_EVALUATION_ROOT} 이 없습니다. "
        "위 목록에서 올바른 run 이름으로 DRIVE_RUN을 수정한 뒤 다시 실행하세요."
    )
epoch_dirs = sorted(DRIVE_EVALUATION_ROOT.glob("epoch-*"))
if not epoch_dirs:
    raise FileNotFoundError(f"{DRIVE_EVALUATION_ROOT} 안에 epoch-* 결과가 없습니다.")
print("Completed epoch evaluations:", [p.name for p in epoch_dirs])

## 시각화 코드 불러오기

`routerarena_visualization.py`는 표준 라이브러리만 사용하므로, torch/transformers 등
무거운 설치 없이 소스만 압축 해제해서 `sys.path`에 추가하면 바로 import할 수 있습니다.
`modernbert_full_colab.ipynb`에서 이미 업로드했던 것과 같은
`modernbert-colab-balanced-v2-input.tar.gz`를 재사용합니다.

In [ ]:
import shutil
import sys
import tarfile
from pathlib import Path

archive = Path("/content/modernbert-colab-balanced-v2-input.tar.gz")
workspace = Path("/content/LLM_Router")
if not archive.is_file():
    raise FileNotFoundError(
        "modernbert_full_colab.ipynb에서 쓴 modernbert-colab-balanced-v2-input.tar.gz를 "
        "Colab /content에 먼저 업로드하세요. 이 notebook은 재학습용 무거운 설치 없이 "
        "그 안의 시각화 코드(ml/src)만 사용합니다."
    )
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with tarfile.open(archive, "r:gz") as bundle:
    bundle.extractall(workspace, filter="data")

src_path = str(workspace / "ml" / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from llm_router_labeling.evaluation.routerarena_visualization import (
    write_routerarena_distribution_charts,
)

print("Loaded visualization module with per-cell counts support.")

## Epoch별 건수 + 퍼센티지 표와 차트

각 epoch의 `metrics.json`을 다시 읽어 predicted-group x RouterArena label 행렬을
건수 표, 비율 표, 그리고 건수가 함께 찍힌 heatmap/stacked bar SVG로 보여줍니다.

재생성한 SVG는 `/content`의 임시 폴더에만 저장하고, Drive에 있는 sealed epoch artifact
(`epoch-0N/`)는 그대로 둡니다 -- 그 폴더는 `modernbert_full_colab.ipynb`가 재개될 때
manifest 무결성을 검사하므로 여기서 덮어쓰면 재개가 깨집니다.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

RENDER_ROOT = Path("/content/routerarena-epoch-charts-with-counts")
RENDER_ROOT.mkdir(parents=True, exist_ok=True)

summary_rows = []
for epoch_dir in epoch_dirs:
    metrics_path = epoch_dir / "metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    epoch_number = metrics["epoch_observation"]["epoch_number"]

    distribution = metrics["routerarena_labels_within_predicted_groups"]
    labels = tuple(metrics["label_order"])

    counts_table = pd.DataFrame(
        {
            group: {
                label: distribution["groups"][group]["counts"][label]
                for label in labels
            }
            for group in labels
        }
    ).T
    counts_table.index.name = "predicted_group"
    counts_table["row_count"] = [
        distribution["groups"][group]["row_count"] for group in labels
    ]

    ratios_table = pd.DataFrame(
        {
            group: {
                label: distribution["groups"][group]["ratios"][label]
                for label in labels
            }
            for group in labels
        }
    ).T
    ratios_table.index.name = "predicted_group"

    print(f"\n=== Epoch {epoch_number} -- accuracy {metrics['accuracy']:.3f}, "
          f"macro F1 {metrics['macro_f1']:.3f} ===")
    print("Counts (columns = RouterArena official label, rows = predicted group):")
    display(counts_table)
    print("Row-normalized ratios:")
    display(ratios_table.style.format("{:.1%}"))

    render_dir = RENDER_ROOT / epoch_dir.name
    stacked_path, heatmap_path = write_routerarena_distribution_charts(
        metrics, render_dir
    )
    display(SVG(filename=str(heatmap_path)))
    display(SVG(filename=str(stacked_path)))

    summary_rows.append(
        {
            "epoch": epoch_number,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "under_routing_rate": metrics["under_routing_rate"],
        }
    )

display(pd.DataFrame(summary_rows).set_index("epoch"))

## 참고

- 건수(count) 자체는 새로 계산한 값이 아니라, 평가 당시 이미
  `metrics.json`의 `routerarena_labels_within_predicted_groups.groups[*].counts`에
  저장돼 있던 값입니다. 이 notebook은 그것을 화면에 노출만 합니다.
- 앞으로 새로 실행되는 RouterArena 평가(`routerarena_visualization.py`를 통해 생성되는
  모든 heatmap/stacked bar)는 이제 기본적으로 건수를 함께 표시합니다.